In [ ]:
import numpy as np
import pandas as pd
import json
import os
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.notebook import tqdm

DATASET_ROOT = Path("../dataset")
POSES_ROOT   = DATASET_ROOT / "poses"
METADATA_DIR = DATASET_ROOT / "metadata"
AUG_ROOT     = DATASET_ROOT / "augmented_tcn"

SIGNS_TARGET = [
    "SOUFFRIR", "AIDER",    "FORT",     "MALADE",   "COEUR",
    "TETE",     "MORT",     "PLEURER",      "NON",   "FROID",
    "MANGER",    "OUI",   "TOMBER", "ACCIDENT", "MARCHER",
    "ENCEINTE", "DORMIR",  "BOIRE",     "CHAUD",  "MEDECIN"
]
SIGN_TO_IDX = {s: i for i, s in enumerate(SIGNS_TARGET)}
NUM_CLASSES  = len(SIGNS_TARGET)
TARGET_T     = 32

TARGET_COUNT          = 1291   # match OUI (majority class)
MAX_VARIANTS_PER_ORIG = 50     # cap per original instance

print(f"Classes: {NUM_CLASSES} | Target per class: {TARGET_COUNT} | Max variants/orig: {MAX_VARIANTS_PER_ORIG}")

In [ ]:
instances = pd.read_csv(DATASET_ROOT / "instances.csv")
face_files = list((POSES_ROOT / "face").glob("*.npy"))
available_ids = {f.stem for f in face_files}

df = instances[instances["id"].isin(available_ids)].copy()
df["label"] = df["sign"].map(SIGN_TO_IDX)
df = df[df["label"].notna()].copy()
df["label"] = df["label"].astype(int)

print(f"Total instances with poses: {len(df)}")
print("\nClass distribution:")
print(df["sign"].value_counts().to_string())

fig, ax = plt.subplots(figsize=(12, 4))
df["sign"].value_counts().plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Instance count per sign (raw)")
ax.set_xlabel("Sign")
ax.set_ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Remove T=0 instances
lengths = {}
for _, row in df.iterrows():
    arr = np.load(POSES_ROOT / "face" / f"{row['id']}.npy")
    lengths[row["id"]] = arr.shape[0]
df["T"] = df["id"].map(lengths)
df_clean = df[df["T"] > 0].reset_index(drop=True)
print(f"Removed {len(df) - len(df_clean)} instances with T=0")
print(f"Remaining: {len(df_clean)}")

# Load official splits
with open(METADATA_DIR / "splits" / "train.json") as f:
    train_ids = set(json.load(f))
with open(METADATA_DIR / "splits" / "test.json") as f:
    test_ids = set(json.load(f))

df_train = df_clean[df_clean["id"].isin(train_ids)].reset_index(drop=True)
df_test  = df_clean[df_clean["id"].isin(test_ids)].reset_index(drop=True)

print(f"\nTrain: {len(df_train)} | Test: {len(df_test)}")
print("\nTrain class distribution:")
print(df_train["sign"].value_counts().to_string())